# Treino do modelo multitarefa no Colab

1. *Runtime → Change runtime type → T4 GPU*.
2. No Google Drive, em `MyDrive/iavc/`, o ficheiro `multitask_data.zip` (criado no Mac, a partir de `data/`):
   `zip -rq multitask_data.zip AffectNetCustom UTKFace multitask/labels.csv`
   **Não enviar `dados_caras/` nem `gallery/`** (dados biométricos dos voluntários).
3. Escolher a corrida na célula de configuração e correr tudo.

Cada corrida fica em `MyDrive/iavc/runs/<RUN>/` (`multitask.pth` gravado a cada melhoria, `config.json`, `multitask.onnx` no fim). Depois: copiar a pasta para `models/multitask/` no Mac e correr `python -m src.evaluation.eval_multitask --run <RUN>`.

In [ ]:
# Configuração da corrida
DRIVE = '/content/drive/MyDrive/iavc'
RUN = 'r50_224_l4'          # nome da pasta da corrida
ARCH = 'resnet50'           # mobilenet | resnet18 | resnet50
SIZE = 224                  # tamanho de entrada
FREEZE = 'layer3'           # ResNet: layer1..layer4 (congela até aí, inclusive), ou '' para treinar tudo

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!test -d iavc_lab3 || git clone -q https://github.com/SamuelCrespo22/iavc_lab3.git
%cd /content/iavc_lab3
!git pull -q && git log --oneline -1
!pip install -q onnxruntime onnx onnxscript

In [ ]:
# Dados: copiar do Drive para o disco local (ler do Drive durante o treino é muito lento)
!test -f data/multitask/labels.csv || (cp {DRIVE}/multitask_data.zip /content/ && mkdir -p data && unzip -q -o /content/multitask_data.zip -d data)
!ls data && wc -l data/multitask/labels.csv

In [ ]:
# As corridas gravam diretamente no Drive: sobrevivem se a sessão do Colab cair
!mkdir -p {DRIVE}/runs models && ln -sfn {DRIVE}/runs models/multitask && ls -la models/

In [ ]:
freeze = f'--freeze-until {FREEZE}' if FREEZE else ''
!python -m src.training.train_multitask --run {RUN} --arch {ARCH} --input-size {SIZE} {freeze} --workers 2

In [ ]:
!ls -la {DRIVE}/runs/{RUN}